In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import dask.array as da
import os
from pycontrails.models.gpat.gpat import GPAT, create_jobs_df, filter_jobs_df, load_fl_df, load_pl_df, load_chem_ds, mc_test, boxm_test

In [2]:
outputs_dir = f"{os.getcwd()}/outputs/"

outputs_dir

'/home/ktait98/pycontrails_kt/pycontrails/models/gpat/outputs/'

In [3]:
# Filter criteria
criteria = {
        "n_ac": 1,
        "rt_fl": (pd.Timedelta(minutes=30), pd.Timedelta(hours=2)),
    }

In [4]:
jobs_df = create_jobs_df(outputs_dir)

jobs_df

,t0_fl,rt_fl,ts_fl,ac_type,fl0_speed,fl0_heading,fl0_coords0,sep_dist,n_ac,dt_integration,...,alt_bounds,hres_sim,vres_sim,eastward_wind,northward_wind,lagrangian_tendency_of_air_pressure,species_in,species_out,date_created,species_out_num
job_id,,,,,,,,,,,,,,,,,,,,,
403478,2022-01-20 13:00:00,0 days 01:00:00,0 days 00:02:00,A320,100.0,45.0,"(0.1, 0.125, 12500)","(5000, 2000, 0)",1,0 days 00:02:00,...,"(12000, 13000)",0.05,500,0.0,0.0,0.0,"[NO, NO2, CO]","[O3, NO2, NO, NO3, HNO3, PAN, HONO, HO2, OH, H...",2024-11-06 17:53:58.975393,"[6, 4, 8, 5, 14, 198, 13, 9, 3, 12, 11, 39, 21]"


In [5]:
filtered_df = filter_jobs_df(jobs_df, criteria)

job_ids = filtered_df.index.values

job_ids

array(['403478'], dtype=object)

In [6]:
fl_df = load_fl_df(job_ids, outputs_dir)

fl_df

,longitude,latitude,altitude,time,air_temperature,specific_humidity,true_airspeed,flight_id,aircraft_mass,engine_efficiency,...,NO,NO2,CO,HCHO,CH3CHO,C2H4,C3H6,C2H2,BENZENE,waypoint
job_id,,,,,,,,,,,,,,,,,,,,,
403478,0.125000,0.100000,12500.0,2022-01-20 13:00:00,212.263516,0.000219,100.230014,0.0,62518.613860,0.099880,...,2.048795,0.107831,0.062205,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,0
403478,0.201229,0.176736,12500.0,2022-01-20 13:02:00,212.259403,0.000218,100.229809,0.0,62376.929746,0.099880,...,2.049022,0.107843,0.062205,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,1
403478,0.277457,0.253473,12500.0,2022-01-20 13:04:00,212.255158,0.000218,100.229514,0.0,62235.235247,0.099881,...,2.049256,0.107856,0.062206,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,2
403478,0.353686,0.330209,12500.0,2022-01-20 13:06:00,212.250782,0.000218,100.229130,0.0,62093.530027,0.099881,...,2.049498,0.107868,0.062206,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,3
403478,0.429914,0.406945,12500.0,2022-01-20 13:08:00,212.246275,0.000217,100.228656,0.0,61951.813748,0.099881,...,2.049747,0.107881,0.062207,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,4
403478,0.506143,0.483682,12500.0,2022-01-20 13:10:00,212.241637,0.000217,100.228093,0.0,61810.086073,0.099882,...,2.050005,0.107895,0.062208,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,5
403478,0.582371,0.560418,12500.0,2022-01-20 13:12:00,212.236868,0.000217,100.227441,0.0,61668.346665,0.099882,...,2.050270,0.107909,0.062208,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,6
403478,0.658600,0.637154,12500.0,2022-01-20 13:14:00,212.231968,0.000216,100.229528,0.0,61526.595186,0.099884,...,2.050540,0.107923,0.062209,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,7
403478,0.734838,0.713886,12500.0,2022-01-20 13:16:00,212.226937,0.000216,100.231527,0.0,61384.831486,0.099885,...,2.050818,0.107938,0.062210,0.000728,0.000243,0.00091,0.000243,0.000243,0.000121,8


In [7]:
pl_df = load_pl_df(job_ids, outputs_dir)

pl_df[pl_df["time"] == pd.Timestamp('2022-01-20 13:58:00')]["age"]

job_id
403478   0 days 01:00:00
403478   0 days 00:58:00
403478   0 days 00:56:00
403478   0 days 00:54:00
403478   0 days 00:52:00
403478   0 days 00:50:00
403478   0 days 00:48:00
403478   0 days 00:46:00
403478   0 days 00:44:00
403478   0 days 00:42:00
403478   0 days 00:40:00
Name: age, dtype: timedelta64[ns]

In [59]:
chem_ds = load_chem_ds(job_ids, outputs_dir)

chem_ds_stacked = chem_ds.stack(
            {"cell": ["level", "longitude", "latitude"]}
        )
chem_ds_stacked = chem_ds_stacked.reset_index("cell")

chem_ds_stacked = chem_ds_stacked.assign_coords(species_out=chem_ds_stacked.attrs["species_out"])


max_emi_cell = chem_ds_stacked["emi"].mean(dim="time").argmax()#.item()

# find cell that has max emissions averaged over time in it
cell_chem_ds = chem_ds_stacked.sel(job_id=job_ids[0], cell=max_emi_cell)
cell_chem_ds["emi"].sel(emi_species="CO")

# Select the emissions for the specified species
emi_data = cell_chem_ds["emi"].sel(emi_species="CO")

# Convert time and emi data to pandas Series
time_series = pd.Series(emi_data["time"].values)
emi_series = pd.Series(emi_data.values)

# Print time and emi values side by side
for time, emi in zip(time_series, emi_series):
    print(f"Time: {time}, EMI: {emi}")


Time: 2022-01-20 12:00:00, EMI: 0.0
Time: 2022-01-20 12:00:20, EMI: 0.0
Time: 2022-01-20 12:00:40, EMI: 0.0
Time: 2022-01-20 12:01:00, EMI: 0.0
Time: 2022-01-20 12:01:20, EMI: 0.0
Time: 2022-01-20 12:01:40, EMI: 0.0
Time: 2022-01-20 12:02:00, EMI: 0.0
Time: 2022-01-20 12:02:20, EMI: 0.0
Time: 2022-01-20 12:02:40, EMI: 0.0
Time: 2022-01-20 12:03:00, EMI: 0.0
Time: 2022-01-20 12:03:20, EMI: 0.0
Time: 2022-01-20 12:03:40, EMI: 0.0
Time: 2022-01-20 12:04:00, EMI: 0.0
Time: 2022-01-20 12:04:20, EMI: 0.0
Time: 2022-01-20 12:04:40, EMI: 0.0
Time: 2022-01-20 12:05:00, EMI: 0.0
Time: 2022-01-20 12:05:20, EMI: 0.0
Time: 2022-01-20 12:05:40, EMI: 0.0
Time: 2022-01-20 12:06:00, EMI: 0.0
Time: 2022-01-20 12:06:20, EMI: 0.0
Time: 2022-01-20 12:06:40, EMI: 0.0
Time: 2022-01-20 12:07:00, EMI: 0.0
Time: 2022-01-20 12:07:20, EMI: 0.0
Time: 2022-01-20 12:07:40, EMI: 0.0
Time: 2022-01-20 12:08:00, EMI: 0.0
Time: 2022-01-20 12:08:20, EMI: 0.0
Time: 2022-01-20 12:08:40, EMI: 0.0
Time: 2022-01-20 12:09:00, E

In [39]:
job_id = job_ids[0]
job_id
# mc = mc_test(job_id, jobs_df, fl_df, pl_df, chem_ds)

# mc

'403478'

In [40]:
cell_chem_ds = boxm_test(job_ids[0], 10, chem_ds)
